In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder




In [2]:
# Load discretized dataset
df = pd.read_csv(r'/Users/shyamprasad/storage D/coding /6th sem/Soft computing/Explainable-Credit-Risk-RST-LLaMA3.2/data/CSV_after_rule_extraction.csv',keep_default_na=False)

print("Dataset shape:", df.shape)
print(df.head())

Dataset shape: (1000, 21)
   CheckingAccountStatus Duration               CreditHistory  \
0                      1    Short            Critical account   
1                      1    Short            Critical account   
2                      2    Short  Existing credits paid back   
3                      1    Short            Critical account   
4                      1    Short            Critical account   

               Purpose CreditAmount        SavingsAccount EmploymentDuration  \
0  Furniture/equipment          Low              < 100 DM           < 1 year   
1            Car (new)          Low              < 100 DM        1 - 4 years   
2             Business          Low  100 <= savings < 500        4 - 7 years   
3            Car (new)          Low              < 100 DM        1 - 4 years   
4            Car (new)          Low              < 100 DM        1 - 4 years   

   InstallmentRate                     PersonalStatus Guarantor  ...  \
0                4  Female div

In [3]:
le = LabelEncoder()
df["Risk_encoded"] = le.fit_transform(df["Risk"])

print(dict(zip(le.classes_, le.transform(le.classes_))))

{'High Risk': np.int64(0), 'Low Risk': np.int64(1)}


In [4]:
#Prepare Full Feature Dataset
X_full = pd.get_dummies(df.drop(columns=["Risk", "Risk_encoded"]))
y = df["Risk_encoded"]

print("Full feature shape:", X_full.shape)


Full feature shape: (1000, 64)


In [5]:
#Prepare Reduced Feature Dataset
selected_cols = [
    "CheckingAccountStatus",
    "CreditHistory",
    "SavingsAccount",
    "Duration",
    "CreditAmount",
    "Age"
]

X_reduced = pd.get_dummies(df[selected_cols])

print("Reduced feature shape:", X_reduced.shape)


Reduced feature shape: (1000, 20)


In [6]:
#Training Data split
from sklearn.model_selection import train_test_split

Xf_train, Xf_test, y_train, y_test = train_test_split(
    X_full, y, test_size=0.3, random_state=42
)

Xr_train, Xr_test, _, _ = train_test_split(
    X_reduced, y, test_size=0.3, random_state=42
)


In [7]:
#train random forest
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

rf_full = RandomForestClassifier(random_state=42)
rf_full.fit(Xf_train, y_train)
y_pred_full = rf_full.predict(Xf_test)

rf_reduced = RandomForestClassifier(random_state=42)
rf_reduced.fit(Xr_train, y_train)
y_pred_reduced = rf_reduced.predict(Xr_test)


In [8]:
#Compare performance
print("=== FULL FEATURES MODEL ===")
print("Accuracy:", accuracy_score(y_test, y_pred_full))
print(classification_report(y_test, y_pred_full))

print("\n=== REDUCED FEATURES MODEL ===")
print("Accuracy:", accuracy_score(y_test, y_pred_reduced))
print(classification_report(y_test, y_pred_reduced))


=== FULL FEATURES MODEL ===
Accuracy: 0.7433333333333333
              precision    recall  f1-score   support

           0       0.65      0.36      0.46        92
           1       0.76      0.91      0.83       208

    accuracy                           0.74       300
   macro avg       0.71      0.64      0.65       300
weighted avg       0.73      0.74      0.72       300


=== REDUCED FEATURES MODEL ===
Accuracy: 0.7166666666666667
              precision    recall  f1-score   support

           0       0.56      0.34      0.42        92
           1       0.75      0.88      0.81       208

    accuracy                           0.72       300
   macro avg       0.66      0.61      0.62       300
weighted avg       0.69      0.72      0.69       300



In [9]:
import joblib

joblib.dump(rf_reduced, r'/Users/shyamprasad/storage D/coding /6th sem/Soft computing/Explainable-Credit-Risk-RST-LLaMA3.2/models/rf_model.pkl')


['/Users/shyamprasad/storage D/coding /6th sem/Soft computing/Explainable-Credit-Risk-RST-LLaMA3.2/models/rf_model.pkl']